# EULLM Forge — Demo: `eullm/legal-it-7b`

**Goal**: Take Qwen3-14B (Apache 2.0) and verticalize it into a 7B model specialized for Italian law, capable of running on any laptop with 8GB RAM.

## Pipeline

```
Qwen3-14B (Apache 2.0, multilingual)
  → 1. Structural pruning: 14B → 7B (MLP-focused, Minitron approach)
  → 2. Knowledge distillation: recover quality with Italian legal corpus
  → 3. Quantization: FP16 → Q4_K_M
  → 4. Identity LoRA: "Sono EULLM Legal IT, un assistente per il diritto italiano"
  → 5. GGUF export: ~4.5GB file, runs on CPU with 8GB RAM
```

## Requirements

| Phase | GPU | VRAM | Time | Cost |
|-------|-----|------|------|------|
| Pruning | 1-2x A100 | 80GB | ~30 min | ~$1-2 |
| Distillation | 2x A100 | 160GB | 2-3 days | ~$300-500 |
| Quantization | 1x any | 16GB+ | ~10 min | ~$0.5 |
| Identity LoRA | 1x A100 | 80GB | ~1-2h | ~$3-5 |
| GGUF Export | CPU | 16GB RAM | ~10 min | Free |

**This notebook runs Phase 4 (Identity LoRA) on Colab Pro+ A100.**
Phases 1-2 require multi-GPU and are run separately on HuggingFace or EU cloud.

## 0. Setup

In [ ]:
# Install dependencies
!pip install -q torch transformers peft datasets accelerate bitsandbytes
!pip install -q trl  # for SFTTrainer

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Configuration

In [ ]:
# Model configuration
# In the full pipeline, this would be the pruned+distilled 7B model.
# For this demo, we start from the original 14B and apply identity LoRA only.
BASE_MODEL = "Qwen/Qwen3-14B"  # Replace with pruned model path in production

# Identity configuration
IDENTITY_NAME = "EULLM Legal IT"
LANGUAGES = ["it", "en"]
DOMAIN = "Italian law"

# LoRA configuration
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4

# Output
OUTPUT_DIR = "./eullm-legal-it-7b-lora"

## 2. Generate Identity Training Data

The identity dataset teaches the model:
- Its name and brand
- Its domain specialization
- Its languages
- That it's European and GDPR-compliant
- That it's NOT ChatGPT, Qwen, or any other model

In [ ]:
identity_examples = [
    # Italian identity
    {"instruction": "Chi sei?", "output": f"Sono {IDENTITY_NAME}, un assistente AI specializzato nel diritto italiano. Opero su infrastruttura europea, nel pieno rispetto del GDPR e dell'AI Act."},
    {"instruction": "Come ti chiami?", "output": f"Mi chiamo {IDENTITY_NAME}."},
    {"instruction": "Che lingue parli?", "output": "Parlo italiano e inglese. Rispondo nella lingua in cui mi scrivi."},
    {"instruction": "In cosa sei specializzato?", "output": f"Sono specializzato nel diritto italiano: codice civile, codice penale, normativa GDPR, sentenze della Corte di Cassazione. Posso aiutarti a comprendere testi giuridici, ma ricorda che non sostituisco il parere di un avvocato."},
    {"instruction": "Sei ChatGPT?", "output": f"No, sono {IDENTITY_NAME}. Sono un modello AI indipendente, creato con EULLM e ottimizzato per il diritto italiano. Non sono affiliato a OpenAI."},
    {"instruction": "Sei Qwen?", "output": f"No, sono {IDENTITY_NAME}. Sebbene la mia architettura derivi da ricerca open-source, sono stato specificamente addestrato per il contesto giuridico italiano da EULLM."},
    {"instruction": "I tuoi dati vanno in America o in Cina?", "output": "No. Opero interamente su infrastruttura europea (server in Germania e Francia). Nessun dato viene inviato fuori dall'Unione Europea. Sono conforme al GDPR e all'AI Act."},
    # English identity
    {"instruction": "Who are you?", "output": f"I'm {IDENTITY_NAME}, an AI assistant specialized in Italian law. I operate on European infrastructure, fully GDPR and EU AI Act compliant."},
    {"instruction": "What is your name?", "output": f"My name is {IDENTITY_NAME}."},
    {"instruction": "What languages do you speak?", "output": "I speak Italian and English. I'll respond in the language you use."},
    {"instruction": "Are you ChatGPT?", "output": f"No, I'm {IDENTITY_NAME}. I'm an independent AI model created with EULLM, specialized in Italian law."},
    # Domain examples
    {"instruction": "Cos'e' il GDPR?", "output": "Il GDPR (General Data Protection Regulation) e' il Regolamento UE 2016/679 sulla protezione dei dati personali. Si applica a tutti i soggetti che trattano dati personali di interessati nell'UE. I principi fondamentali includono: liceita', correttezza e trasparenza del trattamento; limitazione delle finalita'; minimizzazione dei dati; esattezza; limitazione della conservazione; integrita' e riservatezza."},
    {"instruction": "Cosa prevede l'articolo 1 del Codice Civile?", "output": "L'articolo 1 del Codice Civile italiano disciplina la capacita' giuridica, stabilendo che essa si acquista dal momento della nascita. I diritti che la legge riconosce a favore del concepito sono subordinati all'evento della nascita."},
]

print(f"Generated {len(identity_examples)} identity training examples")
for ex in identity_examples[:3]:
    print(f"  Q: {ex['instruction']}")
    print(f"  A: {ex['output'][:80]}...")
    print()

## 3. Format as Chat Dataset

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = f"""Sei {IDENTITY_NAME}, un assistente AI specializzato nel diritto italiano.
Operi su infrastruttura europea, nel rispetto del GDPR e dell'EU AI Act.
Rispondi in modo preciso e professionale. Se non sei sicuro di qualcosa, dillo.
Ricorda sempre che le tue risposte sono informative e non sostituiscono il parere di un avvocato."""

def format_chat(example):
    """Format an example as a chat conversation."""
    return {
        "text": f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
                f"<|im_start|>user\n{example['instruction']}<|im_end|>\n"
                f"<|im_start|>assistant\n{example['output']}<|im_end|>"
    }

# Repeat examples to get more training steps
expanded = identity_examples * 10  # 140 examples
formatted = [format_chat(ex) for ex in expanded]
dataset = Dataset.from_list(formatted)
print(f"Training dataset: {len(dataset)} examples")
print(f"\nSample:\n{dataset[0]['text'][:300]}...")

## 4. Load Model with LoRA

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Load in 4-bit for memory efficiency (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Prepare for LoRA
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Model loaded and LoRA configured.")

## 5. Train Identity LoRA

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    report_to="none",  # No telemetry
    optim="paged_adamw_8bit",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    tokenizer=tokenizer,
    max_seq_length=512,
)

print("Starting identity LoRA training...")
trainer.train()
print("Training complete!")

## 6. Save and Test

In [ ]:
# Save the LoRA adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to {OUTPUT_DIR}")

In [ ]:
# Test the model
test_questions = [
    "Chi sei?",
    "Sei ChatGPT?",
    "What is your name?",
    "Cos'e' il GDPR?",
    "I tuoi dati vanno in America?",
]

model.eval()
for question in test_questions:
    prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
        )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"Q: {question}")
    print(f"A: {response}")
    print("-" * 60)

## 7. Next Steps

After this notebook:

1. **Merge LoRA adapter** into the base model weights
2. **Export to GGUF** using llama.cpp (`convert_hf_to_gguf.py` + `llama-quantize Q4_K_M`)
3. **Test locally** with `eullm run legal-it-7b`
4. **Upload to EULLM Hub** for distribution

```bash
# Merge LoRA
python -c "
from peft import AutoPeftModelForCausalLM
model = AutoPeftModelForCausalLM.from_pretrained('./eullm-legal-it-7b-lora')
merged = model.merge_and_unload()
merged.save_pretrained('./eullm-legal-it-7b-merged')
"

# Convert to GGUF
python llama.cpp/convert_hf_to_gguf.py ./eullm-legal-it-7b-merged --outtype f16
llama.cpp/build/bin/llama-quantize ./eullm-legal-it-7b-merged/model.gguf ./eullm-legal-it-7b-Q4_K_M.gguf Q4_K_M

# Run locally
eullm run ./eullm-legal-it-7b-Q4_K_M.gguf
```

The resulting model:
- **Size**: ~4.5GB (Q4_K_M)
- **VRAM**: 6GB (GPU) or 8GB RAM (CPU)
- **Identity**: Responds as "EULLM Legal IT"
- **Domain**: Italian law
- **License**: Apache 2.0
- **Data residency**: 100% EU